____________________
# Exploring TEI with E Tree

Here we will explore Tasso in Music Project text files encoded in the Text Encoding Initiative (TEI) format. TEI is a widely used standard for representing texts in digital form, particularly in the humanities. TEI files are XML files that contain rich metadata and structural information about texts, making them suitable for various types of analysis.



* **access**: downloading, storing, reading, processing XML files
* **analysis**: performing basic quantitative and qualitative analysis of XML files
* **interpretation**: exploring the meaning and utility of XML files from both analytical and creative perspectives 

As we deal with these files, we will make use of lxml.etree for XML parsing.  


XML files are guided by **markup rules**, which you can read more about [here](https://www.w3schools.com/xml/xml_syntax.asp) and consist of **elements**, which you can dive into [here](https://www.w3schools.com/xml/xml_elements.asp).

TEI files are primarily XML files with specific sections, elements, and structure. You can learn more about TEI [here](https://tei-c.org/).


----

## 1. Import Libraries

In [1]:
import requests
import textwrap
import base64
import pandas as pd
from lxml import etree
import networkx as nx
from pyvis.network import Network
from IPython.display import HTML, display
import pronouncing

import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook"


## 2. Import TEI File



In [2]:
# here we get the file from the Tasso in Music github pages

url = 'https://raw.githubusercontent.com/TassoInMusicProject/tasso-poem-markup/main/data/tei/Tsg16012.tei'

# Load the XML from the web
response = requests.get(url)
xml_content = response.text

# Parse with lxml, using recover=True to handle entity issues
parser = etree.XMLParser(recover=True)
root = etree.fromstring(xml_content.encode('utf-8'), parser=parser)



## 3. Exploring the Header

**Summary: The Four Sections of a Standard `<teiHeader>`**

| Section | Tag | Main Child Elements | Purpose |
|---|---|---|---|
| File Description | `<fileDesc>` | `<titleStmt>`, `<editionStmt>`, `<publicationStmt>`, `<notesStmt>`, `<sourceDesc>` | Who made the file, who published it, and where the original source came from |
| Encoding Description | `<encodingDesc>` | `<editorialDecl>`, `<classDecl>` → `<taxonomy>` → `<category>` | How the document was transcribed and how it is classified |
| Profile Description | `<profileDesc>` | `<creation>`, `<textClass>` → `<keywords>` → `<item>` | When and where the file was created; which subject keywords apply |
| Revision Description | `<revisionDesc>` | `<change>` → `<date>`, `<respStmt>`, `<item>` | Chronological log of every editorial change — who, when, and what |

Note that in the **Tasso in Music Project** TEI files, the `<revisionDesc>` section is not used, and the `<profileDesc>` section is only partially filled out. The `<encodingDesc>` section is also quite minimal, with only a brief editorial declaration and a simple taxonomy for classifying the text as "poetry" and "Italian". The `<fileDesc>` section contains more detailed information about the title, edition, publication, and source of the text.


Here is what the header of the Tasso in Music Project TEI files looks like:

```xml
<teiHeader>
		<fileDesc>
			<titleStmt>
				<title>Vezzosi augelli infra le verdi fronde</title>
			</titleStmt>
			<publicationStmt>
				<publisher>Tasso in Music Project</publisher>
				<idno>Tsg16012</idno>
			</publicationStmt>
            "source": [
                    "rows = []",
                    "lg_num = 0",
                    "for elem in root.iter():",
                    "    if _local_tag(elem) == 'lg':",
                    "        lg_num += 1",
                    "        line_num = 0",
                    "        for child in elem:",
                    "            if _local_tag(child) == 'l':",
                    "                line_num += 1",
                    "                text = ''.join(child.itertext()).strip()",
                    "                rhyme = child.get('rhyme') or ''",
                    "                n_attr = child.get('n') or ''",
                    "                enjamb = child.get('enjamb') or ''",
                    "                rows.append({",
                    "                    'lg': lg_num,",
                    "                    'line': line_num,",
                    "                    'text': text,",
                    "                    'rhyme': rhyme,",
                    "                    'n': n_attr,",
                    "                    'enjamb': enjamb,",
                    "                })",
                    "",
                    "df = pd.DataFrame(rows)",
                    "# extract last word as the rhyme token (clean punctuation)",
                    "df['rhyme_word'] = df['text'].str.split().str[-1].str.strip('.,;:!?\"\'\-')",
                    "df",
                    ""
                ]
```

#### Function to Get fileDesc Information




In [3]:
# function to get the fileDesc

def _find_local(element, local_name):
    for el in element.iter():
        if isinstance(el.tag, str) and el.tag.split('}', 1)[-1] == local_name:
            return el
    return None

def _findall_local(element, local_name):
    return [el for el in element.iter() if isinstance(el.tag, str) and el.tag.split('}', 1)[-1] == local_name]

header = _find_local(root, 'teiHeader')
fd = _find_local(header, 'fileDesc') if header is not None else None

print("=== fileDesc ===")

if fd is None:
    print("No fileDesc found.")
else:
    print("direct children:", [child.tag.split('}', 1)[-1] for child in fd])

    title_stmt = _find_local(fd, 'titleStmt')
    if title_stmt is not None:
        print("\n-- titleStmt --")
        for title in _findall_local(title_stmt, 'title'):
            print(f"  title: {''.join(title.itertext()).strip()}")

    pub = _find_local(fd, 'publicationStmt')
    if pub is not None:
        print("\n-- publicationStmt --")
        for tag in ['publisher', 'pubPlace', 'date']:
            el = _find_local(pub, tag)
            if el is not None:
                print(f"  {tag}: {''.join(el.itertext()).strip()}")
        for idno in _findall_local(pub, 'idno'):
            print(f"  idno: {''.join(idno.itertext()).strip()}")

    src = _find_local(fd, 'sourceDesc')
    if src is not None:
        print("\n-- sourceDesc --")
        for child in src:
            tag = child.tag.split('}', 1)[-1]
            print(f"  {tag}: {''.join(child.itertext()).strip()}")

=== fileDesc ===
direct children: ['titleStmt', 'publicationStmt', 'sourceDesc']

-- titleStmt --
  title: Vezzosi augelli infra le verdi fronde

-- publicationStmt --
  publisher: Tasso in Music Project
  idno: Tsg16012

-- sourceDesc --
  p: Born digital with examples from Tasso in Music Project


In [4]:
# encodingDesc

def _find_local(element, local_name):
    for el in element.iter():
        if isinstance(el.tag, str) and el.tag.split('}', 1)[-1] == local_name:
            return el
    return None

def _findall_local(element, local_name):
    return [el for el in element.iter() if isinstance(el.tag, str) and el.tag.split('}', 1)[-1] == local_name]

ed = _find_local(header, 'encodingDesc') if header is not None else None

print("=== encodingDesc ===")

if ed is None:
    print("No encodingDesc found.")
else:
    print("direct children:", [child.tag.split('}', 1)[-1] for child in ed])
    for child in ed:
        tag = child.tag.split('}', 1)[-1]
        print(f"\n-- {tag} --")
        if tag == 'metDecl':
            for met_sym in _findall_local(child, 'metSym'):
                value = met_sym.get('value', '')
                text = ''.join(met_sym.itertext()).strip()
                print(f"  metSym value={value}: {text}")
        else:
            print(f"  {''.join(child.itertext()).strip()}")

=== encodingDesc ===
direct children: ['metDecl']

-- metDecl --
  metSym value=+: stressed syllable
  metSym value=-: unstressed syllable
  metSym value=_: elision (synalepha?)


In [5]:

# function to get the profileDesc

def _find_local(element, local_name):
    for el in element.iter():
        if isinstance(el.tag, str) and el.tag.split('}', 1)[-1] == local_name:
            return el
    return None

def _findall_local(element, local_name):
    return [el for el in element.iter() if isinstance(el.tag, str) and el.tag.split('}', 1)[-1] == local_name]

profd = _find_local(header, 'profileDesc') if header is not None else None

print("=== profileDesc ===")

if profd is not None:
    creation = _find_local(profd, 'creation')
    if creation is not None:
        date = _find_local(creation, 'date')
        name = _find_local(creation, 'name')
        print("\n-- creation --")
        if date is not None:
            print(f"  date: {''.join(date.itertext()).strip()}")
        if name is not None:
            print(f"  place: {''.join(name.itertext()).strip()}")

    keywords = _findall_local(profd, 'item')
    if keywords:
        print("\n-- textClass / keywords --")
        for item in keywords:
            print(f"  item: {''.join(item.itertext()).strip()}")
else:
    print("No profileDesc found.")

=== profileDesc ===
No profileDesc found.


## 4. Visualizing TEI Structure as a Network

One of the best ways to understand any XML document is to visualize its **tree structure** as an interactive network. In the graphs below:

- Each **node** is a TEI element (a tag like `<teiHeader>` or `<l>`)
- Each **directed edge** points from a parent element to its child
- **Node size** reflects how many descendant elements it contains — larger nodes are structurally richer
- **Node color** encodes tree depth — the root is one color, its children another, and so on

Because TEI files are strictly hierarchical (every element has exactly one parent), these networks are **trees**, not webs. The visual layout makes it easy to see which elements are "hubs" of content and which are leaf nodes.

Use the interactive controls to:
- **Drag** nodes to untangle the layout
- **Scroll** to zoom in and read labels
- **Hover** over a node to highlight its immediate connections

In [6]:
# functions to build a network of elements
def _local_tag(element):
    tag = element.tag
    return tag.split("}", 1)[1] if "}" in tag else tag


def format_element_et(element, wrap_length=20, exclude=[]):
    attrs_list = []
    for a, v in element.attrib.items():
        if a in exclude:
            continue
        attrs_list.append(f"{a}={v}")
    tag_name = _local_tag(element)
    attrs_str = " ".join(attrs_list)
    formatted_string = f"{tag_name} ({attrs_str})" if attrs_list else tag_name
    return textwrap.fill(formatted_string, wrap_length)


def create_network_et(element, with_attributes=False, attrs_to_exclude=[], max_depth=None):
    all_elements = list(element.iter())
    elem_to_idx = {el: i for i, el in enumerate(all_elements)}

    depth_map = {}
    for el in all_elements:
        parent = el.getparent()
        if parent is None or parent not in depth_map:
            depth_map[el] = 0
        else:
            depth_map[el] = depth_map[parent] + 1

    if max_depth is not None:
        all_elements = [el for el in all_elements if depth_map[el] <= max_depth]

    G = nx.DiGraph()

    for node in all_elements:
        depth = depth_map.get(node, 0)
        tag_name = _local_tag(node)
        G.add_node(
            elem_to_idx[node],
            label=format_element_et(node, exclude=attrs_to_exclude) if with_attributes else tag_name,
            value=sum(1 for _ in node.iter()) - 1,
            group=depth,
            level=depth,
            scaling={"label": {"enabled": True}},
        )

    for node in all_elements:
        parent_idx = elem_to_idx[node]
        for child in node:
            child_idx = elem_to_idx.get(child)
            if child_idx is not None and (max_depth is None or depth_map.get(child, 0) <= max_depth):
                G.add_edge(
                    parent_idx, child_idx,
                    arrows="to",
                    id=f"{parent_idx}_{_local_tag(node)}|{child_idx}_{_local_tag(child)}",
                )

    return G


def display_network(network,
                    filename="tmp.html",
                    width="100%",
                    height="650px",
                    bgcolor="white",
                    font_color="black"):
    # cdn_resources="in_line" bundles vis.js into the file -- no CDN needed
    nt = Network(width=width, height=height, bgcolor=bgcolor,
                 font_color=font_color)
    nt.from_nx(network)
    nt.save_graph(filename)
    # Embed as a base64 data-URI iframe -- works in VSCode, JupyterLab, and classic Jupyter
    with open(filename, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    display(HTML(
        f'<iframe src="data:text/html;base64,{b64}" '
        f'width="{width}" height="{height}" frameborder="0"></iframe>'
    ))


### 4a.  Network of Head Elements



- `<fileDesc>` is the largest node because it contains the most nested content: titles, responsibility statements, publication details, and source bibliography all live here.
- `<encodingDesc>` sprouts a huge fan of `<category>` nodes via its `<taxonomy>` — these are the EBBA subject keywords (love, crime, royalty, etc.) used to classify ballads across the archive.
- `<revisionDesc>` repeats a regular pattern: each `<change>` contains `<date>`, `<respStmt>`, and `<item>` — a record of every editorial intervention.

In [7]:
# with attributes
# Full network for each direct child of teiHeader — include attributes and children
header = _find_local(root, 'teiHeader')
if header is None:
    print("No teiHeader found.")
else:
    for child in header:
        tag = _local_tag(child)
        # show attributes (if any) for the direct child
        if child.attrib:
            attrs = ', '.join(f"{k}={v}" for k, v in child.attrib.items())
            print(f"── {tag} ──  attrs: {attrs}")
        else:
            print(f"── {tag} ──")
        # include element attributes in node labels, exclude trivial attrs, and expand two levels
        G = create_network_et(child, with_attributes=True, attrs_to_exclude=['n', 'id'], max_depth=2)
        filename = f"tei_{tag}.html".replace(' ', '_')
        display_network(G, filename=filename, height='600px')

── fileDesc ──


/Users/rfreedma/anaconda3/envs/encoding_music/lib/python3.10/site-packages/IPython/core/display.py:431: UserWarning:

Consider using IPython.display.IFrame instead



── encodingDesc ──


### 4b.  Network of the Complete TEI Document

The full document adds the **`<text>`** element alongside **`<teiHeader>`**, showing the two-part architecture of every TEI file:

```
TEI.2
├── teiHeader   (metadata — who, what, when, where, how it was encoded)
└── text        (the actual content of the document)
```

**Inside `<text>` for this poem:**

| Element | Role |
|---|---|
| `<body>` | The main content area |
| `<lg>` | A line group — one stanza (typically 8 lines) |
| `<l>` | A single line of verse, with metrical markup |
| `<seg type="syl">` | A syllable, tagged with metrical weight (`met="+"` or `met="-"`) |

**What to look for:**

- The `<text>` branch is far larger than `<teiHeader>` — with many `<lg>` stanzas and `<l>` lines, producing many deeply nested `<seg>` leaf nodes (one per syllable).
- Each `<l>` element contains detailed metrical markup via `<seg type="syl">` children, showing the stress pattern of every syllable in the line.
- Note that the `<l>` element includes attributes for `rhyme` (a letter),  `n` (the length of the verse, here expressed as an Italian term for the number of syllables) and `enjamb` (a Boolean for syntactic connection with the following line) 
- Notice how the **metadata tree** and the **text tree** are structurally very different: metadata is **wide and varied** (many different element names), while the text body is **deep and repetitive** (the same `<lg>` / `<l>` / `<seg>` pattern repeated for every line).
- There are also <enjamb>

> **Think about it:** What does this detailed metrical encoding tell us about how the Tasso in Music Project represents not just the *words* but also the *music* encoded in the text?

In [8]:

# Network helpers for TEI 
def _local_tag(element):
    tag = getattr(element, 'tag', None)
    if not isinstance(tag, str):
        return ''
    return tag.split('}', 1)[1] if '}' in tag else tag

def format_element_et(element, wrap_length=20, exclude=None):
    if exclude is None:
        exclude = []
    attrs_list = [f"{a}={v}" for a, v in element.attrib.items() if a not in exclude]
    tag_name = _local_tag(element)
    attrs_str = ' '.join(attrs_list)
    formatted = f"{tag_name} ({attrs_str})" if attrs_list else tag_name
    return textwrap.fill(formatted, wrap_length)

def create_network_et(element, with_attributes=False, attrs_to_exclude=None, max_depth=None):
    """Build a NetworkX DiGraph from an lxml element.
    - with_attributes: include attributes in node labels
    - attrs_to_exclude: list of attribute names to hide from labels
    - max_depth: include nodes with depth <= max_depth (root depth=0)
    """
    if attrs_to_exclude is None:
        attrs_to_exclude = []

    all_elements = [el for el in element.iter() if isinstance(getattr(el, 'tag', None), str)]
    depth_map = {}
    for el in all_elements:
        depth = 0
        p = el.getparent()
        while p is not None:
            depth += 1
            p = p.getparent()
        depth_map[el] = depth

    if max_depth is not None:
        filtered = [el for el in all_elements if depth_map.get(el, 0) <= max_depth]
    else:
        filtered = all_elements

    elem_to_idx = {el: i for i, el in enumerate(filtered)}
    G = nx.DiGraph()

    for node in filtered:
        depth = depth_map.get(node, 0)
        label = format_element_et(node, exclude=attrs_to_exclude) if with_attributes else _local_tag(node)
        G.add_node(
            elem_to_idx[node],
            label=label,
            value=sum(1 for _ in node.iter()) - 1,
            group=depth,
            level=depth,
            scaling={"label": {"enabled": True}},
        )

    for node in filtered:
        parent_idx = elem_to_idx[node]
        for child in node:
            if not isinstance(getattr(child, 'tag', None), str):
                continue
            child_idx = elem_to_idx.get(child)
            if child_idx is not None:
                G.add_edge(
                    parent_idx,
                    child_idx,
                    arrows="to",
                    id=f"{parent_idx}_{_local_tag(node)}|{child_idx}_{_local_tag(child)}",
                )

    return G

# Create and display the full TEI network
G_full = create_network_et(root, with_attributes=True, attrs_to_exclude=['n', 'id'])
display_network(G_full, filename='tei_full_network.html', height='800px')

##  5. Explore the Body:  Line Groups and Lines

- Here we will explore the body of the poem, which in the case of Tasso in Music is particularly rich in analytic detail:
- <lg> elements surround the entire stanza.  This is typical of TEI poems.  
- <l> elements for the individual lines, and include attributes that detail `rhyme`, `n` (number of syllables, expressed as the Italian versification term), and `enjamb` (a Boolean indicating syntactic enjambement with previous verse).  
- The lines (<l>) are in turn made up of individual syllables, which are encoded as <seg> elements, each with its own metrical weight, as defined by special features in the <encodingDesc> above.

Let's have a look!

## 5a. Extracting Lines and Rhyme Words

In [9]:
# Build line-level DF by walking text -> body -> lg -> l
text_el = _find_local(root, "text")
body = _find_local(text_el, "body") if text_el is not None else None

rows = []
if body is None:
    raise RuntimeError("No <text>/<body> element found")

for lg_idx, lg in enumerate([c for c in body if _local_tag(c) == "lg"], start=1):
    for line_idx, l in enumerate([c for c in lg if _local_tag(c) == "l"], start=1):
        line_text = "".join(l.itertext()).strip()
        rows.append({
            "lg": lg_idx,
            "line": line_idx,
            "text": line_text,
            "rhyme": l.get("rhyme", ""),
            "n": l.get("n", ""),
            "enjamb": l.get("enjamb", ""),
        })

df = pd.DataFrame(rows)

df

,lg,line,text,rhyme,n,enjamb
0,1,1,Vez\n\t\t\t\t\tzo\n\t\t\t\t\tsi au\n\t\t\t\t\t...,a,endecasillabo,yes
1,1,2,Tem\n\t\t\t\t\tpra\n\t\t\t\t\tno a \n\t\t\t\t\...,b,endecasillabo,
2,1,3,Mor\n\t\t\t\t\tmo\n\t\t\t\t\tra \n\t\t\t\t\tl’...,a,endecasillabo,yes
3,1,4,"Gar\n\t\t\t\t\trir, \n\t\t\t\t\tche \n\t\t\t\t...",b,endecasillabo,
4,1,5,Quan\n\t\t\t\t\tdo \n\t\t\t\t\ttac\n\t\t\t\t\t...,a,endecasillabo,
5,1,6,Quan\n\t\t\t\t\tdo \n\t\t\t\t\tcan\n\t\t\t\t\t...,b,endecasillabo,
6,1,7,Sia \n\t\t\t\t\tca\n\t\t\t\t\tso od \n\t\t\t\t...,c,endecasillabo,yes
7,1,8,Al\n\t\t\t\t\tter\n\t\t\t\t\tna i \n\t\t\t\t\t...,c,endecasillabo,


In [10]:

def _seg_text(seg):
    return "".join(seg.itertext())

text_el = _find_local(root, "text")
body = _find_local(text_el, "body") if text_el is not None else None
if body is None:
    raise RuntimeError("No <text>/<body> element found")

rows = []
for lg_idx, lg in enumerate([c for c in body if _local_tag(c) == "lg"], start=1):
    for line_idx, l in enumerate([c for c in lg if _local_tag(c) == "l"], start=1):
        segs = [c for c in l if _local_tag(c) == "seg" and c.get("type") == "syl"]
        line_text = "".join(_seg_text(s) for s in segs).strip()
        rows.append({
            "lg":            lg_idx,
            "line":          line_idx,
            "text":          line_text,
            "rhyme":         l.get("rhyme", ""),
            "num_syllables": len(segs),
            "line_type":     l.get("n", ""),
            "enjamb":        l.get("enjamb", ""),
        })

df = pd.DataFrame(rows)
df


,lg,line,text,rhyme,num_syllables,line_type,enjamb
0,1,1,Vezzosi augelli infra le verdi fronde,a,11,endecasillabo,yes
1,1,2,Temprano a prova lascivette note.,b,11,endecasillabo,
2,1,3,"Mormora l’aura, e fa le foglie e l’onde",a,11,endecasillabo,yes
3,1,4,"Garrir, che variamente ella percote:",b,11,endecasillabo,
4,1,5,"Quando taccion gli augelli, alto risponde;",a,11,endecasillabo,
5,1,6,"Quando cantan gli augei, più lieve scote:",b,11,endecasillabo,
6,1,7,"Sia caso od arte, or accompagna ed ora",c,11,endecasillabo,yes
7,1,8,Alterna i versi lor la musica ora.,c,11,endecasillabo,


### 4b. Analyze Rhymes with the `pronouncing` Library

This library works phonetically, so it can identify rhymes even when the spelling is different (e.g., "love" and "dove"). It uses the CMU Pronouncing Dictionary, which is a widely used resource in computational linguistics for English pronunciation.

In [11]:
df['rhyme_word'] = df['text'].str.split().str[-1].str.strip('.,;:!?"\'-')

schemes = {}
for lg_id, group in df.groupby('lg'):
    rhyme_map = {}
    counter = 0
    for idx, word in zip(group.index, group['rhyme_word'].str.lower()):
        key = word[-2:]
        if key not in rhyme_map:
            rhyme_map[key] = chr(ord('A') + counter)
            counter += 1
        schemes[idx] = rhyme_map[key]

df['inferred_scheme'] = pd.Series(schemes)
df['tei_scheme'] = df['rhyme'].str.upper()
df['match'] = df['inferred_scheme'] == df['tei_scheme']

df[['lg', 'line', 'rhyme_word', 'tei_scheme', 'inferred_scheme', 'match']]


,lg,line,rhyme_word,tei_scheme,inferred_scheme,match
0,1,1,fronde,A,A,True
1,1,2,note,B,B,True
2,1,3,l’onde,A,A,True
3,1,4,percote,B,B,True
4,1,5,risponde,A,A,True
5,1,6,scote,B,B,True
6,1,7,ora,C,C,True
7,1,8,ora,C,C,True
